# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to explore and process data from a FAIR² Mental Health Survey Dataset using the `mlcroissant` library following FAIR and AI-readiness best practices.

### Dataset Source
The dataset is defined by a Croissant schema available at the following URL:

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. The metadata provides important information about the dataset's structure, contents, and provenance.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (FAIR2 Mental Health Survey)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset metadata (no subscripting on Dataset object!)
dataset = mlc.Dataset(croissant_url)

# Print metadata summary (always access as attributes or serialize to JSON)
md = dataset.metadata.to_json()
print(f"{md['name']}: {md['description']}")

## 2. Data Overview

Let's review the available record sets, fields, and their `@id` values. All references to entities in this dataset must be by their unique `@id`.

We'll display all record sets, and for the main record set, show its fields and sample records.

In [ ]:
# Display all record sets with their @id and field details
print("Available Record Sets in this Dataset:")
for rec in dataset.metadata.record_sets:
    rec_id = rec['@id']
    name = rec.get('name', '(no name)')
    desc = rec.get('description', '')
    print(f"  - @id: {rec_id}")
    print(f"    name: {name}")
    print(f"    description: {desc}")
    print("    Fields in this record set:")
    for field in rec.get('field', []):
        field_id = field['@id']
        field_name = field.get('name', '(no name)')
        print(f"      - @id: {field_id}  name: {field_name}")

    print()

In [ ]:
# Pick main record set for further exploration (via @id)
# (Assume only one main survey record set; get its @id)
main_record_set = dataset.metadata.record_sets[0]['@id']

# Show 3 sample records (dicts with field @id as keys)
print(f"Sample records from record set: {main_record_set}")
for i, rec in enumerate(dataset.records(record_set=main_record_set)):
    print(rec)
    if i == 2:
        break

## 3. Data Extraction

Let's extract the data for all record sets into pandas DataFrames for analysis.

Each DataFrame's columns correspond to field `@id`s. Use the overviews above to select the relevant fields for analysis.

In [ ]:
# Extract all records from each record set by @id
dataframes = {}
record_sets = [r['@id'] for r in dataset.metadata.record_sets]

for recset_id in record_sets:
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"Loaded DataFrame for record set {recset_id} with columns: {df.columns.tolist()}")

# Display the head of the main record set DataFrame
main_df = dataframes[main_record_set]
print(f"Number of records in main DataFrame: {len(main_df)}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Common EDA steps include filtering rows by numeric values, normalizing numeric columns, and grouping by demographic fields. In this dataset, example numeric fields might include GAD-7, PHQ-9, or PSQ survey scores. All references use field `@id`s.

Let's demonstrate filtering, normalization, and grouping using standardized Croissant field IDs.

In [ ]:
# Find the available numeric field IDs in the main record set
numeric_ids = []
for f in dataset.metadata.record_sets[0]['field']:
    if f.get('dataType', '').lower() in ['integer', 'number', 'float'] or f.get('name', '').lower().startswith('score'):
        numeric_ids.append(f['@id'])
print(f"Numeric fields available: {numeric_ids}")

# Choose GAD-7 score field for demonstration (replace with actual @id!)
numeric_field = None
for f in dataset.metadata.record_sets[0]['field']:
    nm = f.get('name', '').lower()
    if ('gad' in nm or 'phq' in nm or 'psq' in nm) and ('score' in nm):
        numeric_field = f['@id']
        break
# Fallback to first numeric field if not found
if numeric_field is None:
    numeric_field = numeric_ids[0]
print(f"Analyzing numeric field: {numeric_field}")

# Remove missing/invalid rows for demonstration
main_df_numeric = main_df[pd.to_numeric(main_df[numeric_field], errors='coerce').notnull()]
main_df_numeric[numeric_field] = pd.to_numeric(main_df_numeric[numeric_field])

# Filtering: Show records with high value in this field
threshold = 10
filtered_df = main_df_numeric[main_df_numeric[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalization (z-score per field)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records (z-score):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping: Group by a categorical field such as 'village' (replace with @id)
group_field = None
for f in dataset.metadata.record_sets[0]['field']:
    if 'village' in f.get('name', '').lower() or 'location' in f.get('name', '').lower():
        group_field = f['@id']
        break
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Now let's visualize some data distributions, such as histogram of the GAD-7/PHQ-9/PSQ scores and distribution by village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram for the main numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df_numeric[numeric_field], bins=20, kde=True, color='cornflowerblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field is available, show boxplot by category (e.g., village)
if group_field and group_field in main_df_numeric:
    plt.figure(figsize=(9,4))
    sns.boxplot(data=main_df_numeric, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

This notebook guided you through accessing and processing the Kilifi County FAIR² mental health survey dataset using the `mlcroissant` library. We demonstrated metadata exploration, dynamic and robust extraction via `@id`, basic exploratory steps, and simple visualizations, all in accord with AI-ready FAIR principles. You can extend this workflow for advanced analysis and machine learning, knowing that all field-level references will remain stable and interoperable due to the Croissant schema's design.